# 03 · Data checks before re-splitting

Answers three questions before we choose a split:

1. **Are some training copies flipped or rotated?** (Version 4 should be colour-only.)
2. **Are trays cut off by Roboflow's centre crop, or by the camera?** (Version 1 vs Version 4.)
3. **Why do 43 frames have copies with different labels?**

No GPU needed: a normal CPU runtime is fine. Run the cells top to bottom (Runtime → Run all).
Figures are saved to Google Drive in `pork_rasher/data_checks/` for the report.

In [ ]:
# Setup: Drive, code from GitHub, Roboflow key from Colab Secrets
from google.colab import drive, userdata
drive.mount('/content/drive')
FIGS = "/content/drive/MyDrive/pork_rasher/data_checks"
import os; os.makedirs(FIGS, exist_ok=True)

!rm -rf /content/MMA3001_Project
!git clone -q https://github.com/Logan-Das/MMA3001_Project.git /content/MMA3001_Project
!pip install -q roboflow

import sys
sys.path.insert(0, "/content/MMA3001_Project/src")

In [ ]:
# Download Version 4 (used so far) and Version 1 (resized with 'fit', no crop)
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
proj = rf.workspace("hello-2aqe0").project("pork-rasher-error-packaging")
SRC = proj.version(4).download("yolov11", location="/content/v4").location
V1 = proj.version(1).download("yolov11", location="/content/v1").location
print(SRC, V1)

In [ ]:
# Shared helpers
from collections import defaultdict, Counter
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from porkqc.grouping import parse_frame_id
from porkqc.labels import list_images, label_path_for, read_yolo_label

def all_copies(root, splits=("train", "valid", "test")):
    out = defaultdict(list)                      # frame number -> [(split, path)]
    for d in splits:
        for p in list_images(f"{root}/{d}/images"):
            out[parse_frame_id(p.name).frame].append((d, p))
    return out

def thumb(p):
    # 48x36 greyscale, mean 0 / std 1: colour and exposure changes cancel out
    a = np.asarray(Image.open(p).convert("L").resize((48, 36)), dtype=float).ravel()
    return (a - a.mean()) / (a.std() + 1e-9)

def r(a, b):                                     # correlation of two thumbnails
    return float(a @ b) / len(a)

def mirror_r(p, q):                              # correlation if q were flipped left-right
    a = np.asarray(Image.open(q).convert("L").resize((48, 36)), dtype=float)[:, ::-1].ravel()
    return r(thumb(p), (a - a.mean()) / (a.std() + 1e-9))

v4 = all_copies(SRC)
print(len(v4), "unique frames in Version 4")

## Check 1 · Are copies of the same frame flipped or rotated?

Colour-only augmentation keeps copies almost identical (r close to 1). A copy that matches better
**after flipping it back** was mirrored.

In [ ]:
pairs = []                                       # (frame, split_a, split_b, r, r_if_mirrored, path_a, path_b)
for f, cs in v4.items():
    if len(cs) > 1:
        (da, pa) = cs[0]
        ta = thumb(pa)
        for db, pb in cs[1:]:
            pairs.append((f, da, db, r(ta, thumb(pb)), mirror_r(pa, pb), pa, pb))

rr = np.array([x[3] for x in pairs]); mr = np.array([x[4] for x in pairs])
same = rr >= 0.9
mirrored = (rr < 0.9) & (mr >= 0.9)
other = (rr < 0.9) & (mr < 0.9)
print(f"{len(pairs)} copy pairs")
print(f"  near-identical (r >= 0.9):             {same.sum():5d}  ({same.mean():.0%})")
print(f"  mirrored left-right (match once flipped): {mirrored.sum():5d}  ({mirrored.mean():.0%})")
print(f"  something else (rotated / different):  {other.sum():5d}  ({other.mean():.0%})")

def show_pairs(mask, title, fname, n=3):
    idx = np.where(mask)[0][:n]
    if len(idx) == 0:
        print("  none to show for", title); return
    fig, axes = plt.subplots(len(idx), 2, figsize=(8, 3 * len(idx)), squeeze=False)
    for row, i in zip(axes, idx):
        f, da, db, rv, mv, pa, pb = pairs[i]
        for ax, p, d in zip(row, (pa, pb), (da, db)):
            ax.imshow(Image.open(p)); ax.axis("off")
            ax.set_title(f"frame {f} · {d} copy · r={rv:.2f}", fontsize=9)
    fig.suptitle(title); plt.tight_layout()
    plt.savefig(f"{FIGS}/{fname}", dpi=120); plt.show()

show_pairs(mirrored, "Copies that are mirror images", "copies_mirrored.png")
show_pairs(other, "Copies that differ in another way", "copies_other.png")

## Check 2 · Is the tray cut off by Roboflow's crop or by the camera?

Validation and test images are not augmented in either version, so the only difference is the resize:
Version 1 *fits* the whole picture (white bars), Version 4 *fills* it and crops the centre.

In [ ]:
def valtest(root):
    out = {}
    for d in ["valid", "test"]:
        for p in list_images(f"{root}/{d}/images"):
            out[parse_frame_id(p.name).frame] = p
    return out

a1, a4 = valtest(V1), valtest(SRC)
both = sorted(set(a1) & set(a4))
print(len(both), "frames in valid/test of both versions")
if both:
    print("Version 1 image size:", Image.open(a1[both[0]]).size, "| Version 4 image size:", Image.open(a4[both[0]]).size)

    # How much of the V1 image is white padding? (rows/columns that are almost pure white)
    g = np.asarray(Image.open(a1[both[0]]).convert("L"), dtype=float)
    white_rows = (g.mean(axis=1) > 245).sum(); white_cols = (g.mean(axis=0) > 245).sum()
    print(f"Version 1 white padding: {white_rows} rows, {white_cols} columns")

    pick = both[:: max(1, len(both) // 4)][:4]
    fig, axes = plt.subplots(len(pick), 2, figsize=(9, 3.4 * len(pick)), squeeze=False)
    for row, f in zip(axes, pick):
        for ax, (name, src) in zip(row, [("V1 fit (white edges)", a1), ("V4 fill (centre crop)", a4)]):
            ax.imshow(Image.open(src[f])); ax.set_title(f"{name} · frame {f}", fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.savefig(f"{FIGS}/v1_vs_v4_crop.png", dpi=120); plt.show()

## Check 3 · The frames whose copies have different labels

In [ ]:
odd = []
for f, cs in v4.items():
    labs = [(d, tuple(sorted(read_yolo_label(label_path_for(p))[:, 0].astype(int).tolist()))) for d, p in cs]
    if len({l for _, l in labs}) > 1:
        odd.append((f, labs, cs))
print(len(odd), "frames whose copies carry different labels")
print("  splits involved:", Counter(tuple(sorted({d for d, _ in labs})) for _, labs, _ in odd))
for f, labs, _ in odd[:6]:
    print("  frame", f, labs)

if odd:
    f, labs, cs = odd[0]
    fig, axes = plt.subplots(1, len(cs), figsize=(3.5 * len(cs), 3), squeeze=False)
    for ax, (d, p), (_, l) in zip(axes[0], cs, labs):
        ax.imshow(Image.open(p)); ax.axis("off"); ax.set_title(f"{d} · classes {list(l)}", fontsize=9)
    fig.suptitle(f"frame {f}: copies with different labels"); plt.tight_layout()
    plt.savefig(f"{FIGS}/label_mismatch_example.png", dpi=120); plt.show()

## Check 4 · A run of consecutive frames (for choosing `max_gap`)

Uses the un-augmented valid/test copy where one exists. Look at whether the **same rashers** appear in neighbouring frames.

In [ ]:
path_of = {}
for f, cs in v4.items():
    vt = [p for d, p in cs if d != "train"]
    path_of[f] = vt[0] if vt else cs[0][1]
fr = sorted(path_of)

def show_run(start, n=12, fname=None):
    sel = [f for f in fr if f >= start][:n]
    fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(3 * ((n + 1) // 2), 5))
    for ax, f in zip(axes.ravel(), sel):
        src = "train" if path_of[f].parts[-3] == "train" else path_of[f].parts[-3]
        ax.imshow(Image.open(path_of[f])); ax.set_title(f"frame {f} ({src})", fontsize=9); ax.axis("off")
    plt.tight_layout()
    if fname: plt.savefig(f"{FIGS}/{fname}", dpi=120)
    plt.show()

show_run(3360, fname="run_3360.png")
show_run(400, fname="run_400.png")

## Send back to Claude
1. The printed lines from Checks 1, 2 and 3.
2. Screenshots of the figures (or say what you see). They are also saved in Drive → `pork_rasher/data_checks/`.